# Intention Collapse - Experimento Piloto en Colab

**GPU:** A100  
**Objetivo:** Correr experimento pequeño (50-100 problemas) para validar pipeline

## 1. Setup: Clonar repo e instalar dependencias

In [6]:
# Verificar GPU
!nvidia-smi

Wed Jan  7 14:26:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
import os
from pathlib import Path
import shutil

# Borrar repo si existe
if Path('intention-collapse-experiments').exists():
    shutil.rmtree('intention-collapse-experiments')

# Clonar
!git clone https://github.com/patriciomvera/intention-collapse-experiments.git

# Navegar al directorio CORRECTO (con el duplicado)
os.chdir('intention-collapse-experiments/intention-collapse-experiments')

# Verificar
assert Path('src/shared_utils.py').exists(), "❌ shared_utils.py no encontrado"
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ shared_utils.py encontrado y listo")

✓ Repo anterior eliminado
Cloning into 'intention-collapse-experiments'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 60 (delta 18), reused 41 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 353.35 KiB | 11.04 MiB/s, done.
Resolving deltas: 100% (18/18), done.


AssertionError: ❌ shared_utils.py no encontrado

In [ ]:
# Instalar dependencias
!pip install -q torch transformers datasets accelerate bitsandbytes scikit-learn pandas numpy matplotlib seaborn tqdm pyyaml

## 2. Montar Google Drive (para guardar outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_BASE = Path('/content/drive/MyDrive/intention_collapse_outputs')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print(f"✓ Outputs se guardarán en: {OUTPUT_BASE}")

## 3. Imports y configuración

In [ ]:
import sys
from pathlib import Path

# Agregar directorio actual al path
sys.path.insert(0, str(Path.cwd()))

print(f"Working directory: {Path.cwd()}")
print(f"Python path[0]: {sys.path[0]}")

# Verificar que src/shared_utils.py existe
shared_utils_path = Path('src/shared_utils.py')
if not shared_utils_path.exists():
    raise FileNotFoundError(
        f"❌ No se encuentra src/shared_utils.py\n"
        f"Working dir: {Path.cwd()}\n"
        f"Contenido de src/: {list(Path('src').glob('*')) if Path('src').exists() else 'src/ no existe'}"
    )

# Importar shared_utils
from src import shared_utils as U
print(f"✓ shared_utils.py v{U.__version__} cargado")

# Otros imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import json
from tqdm.auto import tqdm

print(f"✓ GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU no disponible - verifica Runtime settings")

## 4. Cargar modelo y dataset

In [ ]:
# Configuración
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
SAMPLE_SIZE = 50  # Piloto pequeño
LAYERS = [27, 28, 29, 30, 31]  # Últimas capas

# Cargar modelo (4-bit)
print("Cargando modelo...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Modelo cargado: {MODEL_NAME}")

In [ ]:
# Cargar GSM8K
print("Cargando GSM8K...")
dataset = load_dataset("openai/gsm8k", "main", split="test")
sample_dataset = dataset.select(range(SAMPLE_SIZE))

print(f"✓ Dataset cargado: {SAMPLE_SIZE} problemas")

## 5. Definir prompts de los regímenes

In [ ]:
PROMPTS = {
    "baseline": "Solve this math problem. Give only the final numerical answer.\n\nProblem: {question}\n\nAnswer:",
    "cot": "Solve this math problem step by step. Show your reasoning, then give the final answer after ####.\n\nProblem: {question}\n\nSolution:",
    "babble": "Given this math problem, write a long stream of consciousness about numbers and calculations. Do NOT solve it.\n\nProblem: {question}\n\nStream:"
}

MAX_TOKENS = {
    "baseline": 50,
    "cot": 512,
    "babble": 512
}

## 6. Configurar regímenes y funciones auxiliares

In [ ]:
# Ya tenemos todo en shared_utils, solo necesitamos configurar los regímenes
print("✓ Usando funciones de shared_utils.py")
print(f"  - U.extract_activations_with_hooks()")
print(f"  - U.compute_intention_entropy()")
print(f"  - U.compute_effective_dimensionality()")
print(f"  - U.extract_predicted_answer()")
print(f"  - U.check_correctness()")

## 7. Correr experimento piloto

In [ ]:
results = []

for idx, item in enumerate(tqdm(sample_dataset, desc="Procesando problemas")):
    question = item['question']
    ground_truth = U.extract_ground_truth(item['answer'])

    item_results = {
        'idx': idx,
        'question': question,
        'ground_truth': ground_truth,
        'regimes': {}
    }

    # Probar cada régimen
    for regime in ['baseline', 'cot']:  # Omitimos babble para ir más rápido
        prompt = PROMPTS[regime].format(question=question)
        max_tok = MAX_TOKENS[regime]

        try:
            # Usar funciones de shared_utils
            activations, output, first_logits = U.extract_activations_with_hooks(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                layer_indices=LAYERS,
                max_new_tokens=max_tok,
                temperature=0.0,
                return_first_step_only=True
            )

            # Calcular métricas
            H_int = U.compute_intention_entropy(first_logits)
            dim_eff = U.compute_effective_dimensionality(activations, per_layer=False)

            # Extraer respuesta
            predicted = U.extract_predicted_answer(output, regime=regime)
            correct = U.check_correctness(predicted, ground_truth)

            item_results['regimes'][regime] = {
                'output': output,
                'predicted_answer': predicted,
                'correct': correct,
                'intention_entropy': H_int,
                'dim_eff_global': dim_eff.get('global', float('nan')),
                'output_length': len(tokenizer.encode(output))
            }

        except Exception as e:
            print(f"Error en {idx}/{regime}: {e}")
            item_results['regimes'][regime] = {'error': str(e)}

    results.append(item_results)

    # Checkpoint cada 10 problemas
    if (idx + 1) % 10 == 0:
        checkpoint_path = OUTPUT_BASE / f'checkpoint_{idx+1}.json'
        with open(checkpoint_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"✓ Checkpoint guardado: {checkpoint_path}")

print("\n✅ Experimento completado!")

## 8. Guardar resultados finales

In [ ]:
final_path = OUTPUT_BASE / 'pilot_results.json'
with open(final_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Resultados guardados en: {final_path}")
print(f"Problemas procesados: {len(results)}")

## 9. Preview de resultados

In [ ]:
# Ver primer resultado
print("Ejemplo de resultado:")
print(json.dumps(results[0], indent=2))